# Supervised Learning Models

In this section, I evaluate multiple supervised learning models to classify news articles as fake or true using embedding-based features.

Unlike unsupervised learning, which explores the underlying structure of the data without labels, supervised learning leverages labeled data to learn a mapping from input features to a target variable.

---

## Models Used in This Project

The following models are implemented and compared:

### 1. Logistic Regression (Baseline)
- Uses embedding features only
- Serves as a simple linear baseline model

### 2. Logistic Regression (Stacked Features)
- Combines embeddings with **GMM cluster probabilities**
- Incorporates information from unsupervised structure

### 3. Logistic Regression (Tuned)
- Uses hyperparameter tuning (GridSearchCV)
- Optimizes model performance

### 4. Random Forest
- Ensemble-based model
- Captures non-linear relationships

### 5. Gradient Boosting
- Sequential ensemble model
- Focuses on improving difficult predictions

---

## Evaluation Metrics

Models are evaluated using:

- **AUC (Area Under the ROC Curve)**
- **Accuracy**
- **Precision, Recall, and F1-score**

---

## Goal of This Section

The goal is to:

- Compare model performance across different approaches
- Examine whether more complex models improve results
- Understand why supervised models achieve high accuracy
- Connect these results back to the unsupervised findings

---

## Key Question

> If unsupervised analysis shows overlapping structure, why do supervised models achieve near-perfect performance?

---

## Data Splitting Strategy

To avoid data leakage, I use group-based splitting to ensure that highly similar or duplicated articles do not appear in both training and test sets.

This ensures that model performance reflects generalization rather than memorization.

In [ ]:
# Supervised model with probability_gmm
def normalize_text_for_grouping(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df_sub["text_group"] = df_sub["text"].apply(normalize_text_for_grouping)

df_sub["group_id"] = df_sub["text_group"].astype("category").cat.codes

# 1. labels
y = df_sub["label"].values

# 2. test/train split
X = X_embed
y = df_sub["label"].values
groups = df_sub["group_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_base, X_test_base = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

df_train = df_sub.iloc[train_idx].copy()
df_test = df_sub.iloc[test_idx].copy()

print(X_train_base.shape, X_test_base.shape)
print(df_train.shape, df_test.shape)

# 3. fit GMM on training embeddings
gmm = GaussianMixture(
    n_components=10,
    covariance_type="diag",
    random_state=42
)
gmm.fit(X_train_base)

# 4. get GMM probabilities separately
probs_train = gmm.predict_proba(X_train_base)
probs_test = gmm.predict_proba(X_test_base)

# 5. build stacked features (embedding feature & GMM probabilities)
X_train_stack = np.hstack([X_train_base, probs_train])
X_test_stack = np.hstack([X_test_base, probs_test])

# 6. baseline model
base_clf = LogisticRegression(max_iter=2000)
base_clf.fit(X_train_base, y_train)

base_probs = base_clf.predict_proba(X_test_base)[:, 1]
base_preds = base_clf.predict(X_test_base)

print("Baseline AUC:", roc_auc_score(y_test, base_probs))
print(classification_report(y_test, base_preds))
#Baseline AUC: 0.9826318260053202
#              precision    recall  f1-score   support
#           0       0.93      0.92      0.93       539
#           1       0.94      0.94      0.94       664
#    accuracy                           0.94      1203
#   macro avg       0.93      0.93      0.93      1203
#weighted avg       0.94      0.94      0.94      1203

#============================================================================
# 7. stacked model
stack_clf = LogisticRegression(max_iter=2000)
stack_clf.fit(X_train_stack, y_train)

stack_probs = stack_clf.predict_proba(X_test_stack)[:, 1] # fake
stack_preds = stack_clf.predict(X_test_stack)

print("Stacked AUC:", roc_auc_score(y_test, stack_probs))
print(classification_report(y_test, stack_preds))

#Stacked AUC: 0.9801981581241478
#              precision    recall  f1-score   support
#           0       0.93      0.92      0.92       539
#           1       0.93      0.94      0.94       664
#    accuracy                           0.93      1203
#   macro avg       0.93      0.93      0.93      1203
#weighted avg       0.93      0.93      0.93      1203

# 8. grid search on stacked model
param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["lbfgs", "liblinear"]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=2000),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train_stack, y_train)

print("Best params:", grid.best_params_)
print("Best CV AUC:", grid.best_score_)

#Best params: {'C': 10, 'solver': 'liblinear'}
#Best CV AUC: 0.9948587075400745

# 9. final test evaluation
best_clf = grid.best_estimator_
best_probs = best_clf.predict_proba(X_test_stack)[:, 1]
best_preds = best_clf.predict(X_test_stack)

print("Test AUC:", roc_auc_score(y_test, best_probs))
print(classification_report(y_test, best_preds))
#Test AUC: 0.9923553210988667
#              precision    recall  f1-score   support
#           0       0.96      0.95      0.96       539
#           1       0.96      0.97      0.96       664
#    accuracy                           0.96      1203
#   macro avg       0.96      0.96      0.96      1203
#weighted avg       0.96      0.96      0.96      1203

print("Best CV AUC:", grid.best_score_)

#===========================================================================
# Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

rf.fit(X_train_stack, y_train)

rf_probs = rf.predict_proba(X_test_stack)[:, 1]

print("RF AUC:", roc_auc_score(y_test, rf_probs))
# RF AUC: 0.941112781366654

#===========================================================================
# Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

gb.fit(X_train_stack, y_train)

gb_probs = gb.predict_proba(X_test_stack)[:, 1]
print("GB AUC:", roc_auc_score(y_test, gb_probs))
# GB AUC: 0.9603544046315131

#Model                             AUC
#Unsupervised risk score          ~0.775
#Logistic (embedding)             ~0.982  (baseline)
#Logistic (tuned)                  0.992
#Random Forest                     0.941
#Gradient Boosting                 0.960

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, best_preds))
# Accuracy: 0.9609310058187863

### Baseline Model: Logistic Regression

Using embeddings alone, logistic regression achieves very high performance.

## Stacked Model: Embedding + GMM Probabilities

Adding GMM features does not significantly improve performance, suggesting that embeddings already capture most of the predictive signal.

## Model Comparison

|---------Model----------|---AUC---|

|------------------------|---------|

|  Unsupervised risk score | ~0.775 |

|  Logistic (embedding)    | ~0.982  |

**|  Logistic (tuned)        | ~0.992 |**

|  Random Forest           | ~0.941 |

|  Gradient Boosting       | ~0.960 |

### Key Observations

- Simple linear models perform best
- Complex models do not improve performance
- Embeddings dominate predictive power

## Interpretation

Despite the strong performance of supervised models, earlier unsupervised analysis showed that fake and true news are not cleanly separable.

This suggests that:

- The model is not discovering natural clusters
- Instead, it is learning decision boundaries that exploit subtle patterns
- These patterns may include:
  - writing style
  - topic distribution
  - source characteristics

### Key Insight

High classification accuracy does not imply true semantic separation.

Instead, it reflects the model’s ability to leverage structured patterns in the dataset.